# Kiểm tra sơ bộ dataset

Trước khi bắt đầu bất kỳ phân tích nào thì cần kiểm tra dataset, nhóm đã xây dựng một pipeline kiểm tra theo từng module đặt trong src và có thể chạy bằng lệnh:
```bash
cd ..
python src/pipeline.py
```
Kết quả sẽ được lưu vào trong data/validation_report.json. Hoặc chạy notebook này.

In [1]:
from pprint import pprint
import sys
sys.path.insert(0, "../src")

# Kiểm tra cấu trúc

Thực hiện kiểm tra số cột, tên các cột và kiểu dữ liệu các cột của bộ dữ liệu nạp vào dưới dạng DataFrame

In [2]:
from ingestion import load_netflix_titles
from audit import audit_structure

df = load_netflix_titles()
pprint(audit_structure(df))

{'column_names': ['show_id',
                  'type',
                  'title',
                  'director',
                  'cast',
                  'country',
                  'date_added',
                  'release_year',
                  'rating',
                  'duration',
                  'listed_in',
                  'description'],
 'columns': 12,
 'dtypes': {'cast': 'str',
            'country': 'str',
            'date_added': 'str',
            'description': 'str',
            'director': 'str',
            'duration': 'str',
            'listed_in': 'str',
            'rating': 'str',
            'release_year': 'int64',
            'show_id': 'str',
            'title': 'str',
            'type': 'str'},
 'rows': 8807}


Kết quả cho thấy các trường dữ liệu khớp với mô tả trên Kaggle và không có gì sai lệch, có các trường đáng chú ý là show_id nghi ngờ là khóa chính, date_added là ngày thêm vào nghi ngờ là ngày tháng. 

Các trường khác có thể là thuộc tính đa trị như cast, country, director, listed_in lần lượt đại diện cho diễn viên, quốc gia sản xuất, đạo diễn, thể loại. Một phim có thể có nhiều diễn viên, đến từ nhiều quốc gia, có nhiều đạo diễn và thuộc nhiều thể loại.

# Kiểm tra số dòng lặp

Kiểm tra số dòng lặp và xử lý để đảm bảo dữ liệu sạch sẽ

In [3]:
from audit import audit_duplicate

pprint(audit_duplicate(df))

0


# Kiểm tra khóa chính

Kiểm tra xem trường được nghi vấn có đúng là trường khóa chính của bộ dữ liệu hay không

In [4]:
from audit import audit_key

pprint(audit_key(df, "show_id"))

{'duplicate_count': 0, 'key': 'show_id', 'null_count': 0, 'unique_count': 8807}


# Kiểm tra xem có trường nào có thuộc tính đa trị hay không

Kiểm tra các trường có thuộc tính đa trị được phân cách bởi dấu phẩy để chuẩn hóa và tách thành các bảng theo đúng nhu cầu

In [5]:
from audit import audit_multivalue

pprint(audit_multivalue(df))

{'cast': {'average_values_per_row': 8.033826108744675,
          'max_values_per_row': 50,
          'multivalue_rows': 7101,
          'non_null_count': 7982,
          'single_value_rows': 881},
 'country': {'average_values_per_row': 1.2561434302908727,
             'max_values_per_row': 12,
             'multivalue_rows': 1320,
             'non_null_count': 7976,
             'single_value_rows': 6656},
 'date_added': {'average_values_per_row': 2.0,
                'max_values_per_row': 2,
                'multivalue_rows': 8797,
                'non_null_count': 8797,
                'single_value_rows': 0},
 'description': {'average_values_per_row': 2.0124900647212445,
                 'max_values_per_row': 8,
                 'multivalue_rows': 6448,
                 'non_null_count': 8807,
                 'single_value_rows': 2359},
 'director': {'average_values_per_row': 1.1304066094281549,
              'max_values_per_row': 13,
              'multivalue_rows': 614,
        

# Kiểm tra các trường ngày tháng

Kiểm tra các trường dữ liệu nghi ngờ là ngày tháng xem nó có đúng là ngày tháng không và có dòng nào bị lỗi ngày tháng hay không.

In [6]:
from audit import audit_date

pprint(audit_date(df))

{'date_added': {'invalid_count': 0,
                'max_date': '2021-09-25',
                'min_date': '2008-01-01',
                'null_count': 10,
                'original_dtype': 'str'}}


# Kiểm tra phạm vi ngày tháng

Kiểm tra số lượng dòng thuộc phạm vi ngày tháng nghiên cứu của nhóm là 2011-2020

In [7]:
from audit import audit_scope

pprint(audit_scope(df))

{'end_year': 2020,
 'in_scope_percentage': 82.82048370614284,
 'in_scope_rows': 7294,
 'missing_date_rows': 10,
 'out_of_scope_rows': 1503,
 'out_of_scope_years': [2008.0, 2009.0, 2010.0, 2021.0],
 'scope_column': 'date_added',
 'start_year': 2011,
 'total_rows': 8807}
